# KOMPAS   

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import csv
import time
import requests
from bs4 import BeautifulSoup
import re
import os
from urllib.parse import urlparse, parse_qs
import time


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

headers = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win 64; x64) AppleWebKit/537.36 (KHTML, Like Gecko) Chrome/140.0.0.0 Safari/537.36 Edg/140.0.0.0'
}

In [3]:
#setup for csv (write)
CSV_HEADERS = ['title', 'tanggal', 'content']

#setup for csv (append)
def append_rows_to_csv(filename, headers, rows):
    """
    Append baris ke CSV. Header hanya ditulis jika file belum ada atau header belum ada.
    Force flush+fsync di dalam blok 'with' supaya tidak ada operasi setelah file ditutup.
    """
    try:
        # Normalisasi rows (kalau generator/iterator, materialize biar aman)
        rows_to_write = list(rows)
        if not rows_to_write:
            print("Tidak ada baris untuk ditulis.")
            return

        write_header = True
        if os.path.exists(filename):
            # Cek apakah header sudah ada
            try:
                with open(filename, 'r', newline='', encoding='utf-8') as rf:
                    reader = csv.reader(rf)
                    first = next(reader, None)
                    write_header = (first != headers)
            except Exception:
                # Kalau gagal baca (file kosong/korup), kita tulis header lagi saja
                write_header = True

        # Tulis append
        with open(filename, 'a', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            if write_header:
                w.writerow(headers)
            w.writerows(rows_to_write)

            # Pastikan buffer flush sebelum keluar dari 'with'
            f.flush()
            try:
                os.fsync(f.fileno())
            except Exception:
                # fsync bisa gagal di beberapa FS; aman diabaikan
                pass

        print(f"💾 APPEND: {len(rows_to_write)} baris disimpan ke '{filename}'")

    except Exception as e:
        print(f"❌ ERROR APPEND CSV '{filename}': {e}")

START SCRAPING KOMPAS

In [4]:
# baca link berita
def get_article_kompas(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='read__content') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error artikel di: {link} | {e}")
  return ''


In [5]:
# url list
urlkompas = 'https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all'
respkompas = requests.get(urlkompas)
soupkompas = BeautifulSoup(respkompas.text, 'lxml')

# print(soupkompas.text)


In [6]:
#cari lastpage kompas
last_page_kompas = None

# 1) coba langsung selector 'Last' (class paging__link--last)
last_a = soupkompas.select_one('a.paging__link--last')

def extract_page_from_href(href):
    if not href:
        return None
    # coba regex ?page=123 atau &page=123
    m = re.search(r'[?&]page=(\d+)', href)
    if m:
        return int(m.group(1))
    # fallback parse query
    qs = parse_qs(urlparse(href).query)
    if 'page' in qs and qs['page']:
        try:
            return int(qs['page'][0])
        except ValueError:
            return None
    return None

if last_a:
    # prioritas: data-ci-pagination-page attribute
    data_page = last_a.get('data-ci-pagination-page')
    if data_page and data_page.isdigit():
        last_page_kompas = int(data_page)
    else:
        # extract dari href
        last_page_kompas = extract_page_from_href(last_a.get('href'))

print("Halaman terakhir terdeteksi:", last_page_kompas)

Halaman terakhir terdeteksi: 499


In [7]:
# scraping

counter = 0
kompas_batch = []
CSV_FILENAME = 'kompas_politics_articles.csv'
BATCH_SIZE = 1000

#scraping main function
for page in range(1, last_page_kompas+1):
  print(f'scraping hlmn ke - {page}')
  url = f'https://search.kompas.com/search?q=politik+indonesia&sort=latest&site_id=1&last_date=all&page={page}'
  try:
    res = requests.get(url, headers=headers, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, 'lxml')
    articles = soup.find_all('a', class_='article-link') #sesuaiin ke website 

    if not articles:
      print(f"no articles found on page {page}")
      break
    
    for art in articles:
      try:

        # a_tag = art.find('a') #sesuaiin ke website (di kompas ini ngebungkus berita, kek header gt)
        title = art.find('h2').text.strip() #sesuaiin ke website (di kompas ini tag tag an buat judul)
        link = art['href'] #sesuaiin ke website (di cnbc ini buat tembak site beritanya)
        time_info = art.find('div', class_='articlePost-date').text.strip() #sesuaiin ke website (di cnbc ini tag untuk )
        content = get_article_kompas(link)

        # save ke csv
        kompas_batch.append([title, time_info, content])
        

        print(f'✅ {counter} - {title} SCRAPPED!')
        print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]} \n Tanggal artikel: {time_info}')
        counter += 1
      
        if len(kompas_batch) >= BATCH_SIZE:
          append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, kompas_batch)
          kompas_batch.clear()
      except Exception as e:
        print(f"Error parsing artikel di halaman: {page} | {e}")
        continue

  except Exception as e:
    print(f"Error scraping di halaman: {page} | {e}")
    continue

if kompas_batch:
    append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, kompas_batch)
    kompas_batch.clear()

scraping hlmn ke - 1
✅ 0 - Temui 8 Penerima Beasiswa KKS ke Portugal, Puan Dukung Pembinaan Sepak Bola Muda Indonesia SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: KOMPAS.com– Ketua Dewan Perwakilan Rakyat (DPR) RI Puan Maharani bertemu dengan delapan pemain sepak 
 Tanggal artikel: 8 Oktober 2025
✅ 1 - Prabowo ke Patrick Kluivert dan Timnas Indonesia: Beri Kabar Baik Malam Ini! SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com -Presiden RI Prabowo Subianto meminta pelatih Tim Nasional Indonesia Patrick Klu 
 Tanggal artikel: 8 Oktober 2025
✅ 2 - Komnas HAM Soroti Gonta-ganti Kurikulum Pendidikan di Indonesia SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com- Komisi Nasional Hak Asasi Manusia (Komnas HAM) memberikan nilai 66,9 untuk Keme 
 Tanggal artikel: 8 Oktober 2025
✅ 3 - Prabowo "Video Call" Patrick Kluivert dan Timnas Indonesia Jelang Lawan Arab Saudi SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA, KOMPAS.com -Presiden RI Prabowo Subian

KeyboardInterrupt: 

In [ ]:
scrapped = pd.read_csv('kompas_politics_articles.csv')
print(f"\n🎉 DONNN: total artikel KOMPAS yang dah ke scraping: {scrapped.index[-1] if not scrapped.empty else None}")